In [ ]:
from pathlib import Path
import sys
import os
import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modules.dataset_processing import extract_archives, prepare_all_split_dataframes
from modules.audio_processing import extract_prosodic_features, PROSODIC_FEATURE_COLUMNS

DATA_ROOT = PROJECT_ROOT / 'ASVspoof5'
CACHE_DIR = DATA_ROOT / 'feature_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)



N_JOBS = max(1, os.cpu_count() - 2)
print(f"Using {N_JOBS} parallel workers")
print(f"Feature count: {len(PROSODIC_FEATURE_COLUMNS)}")


In [ ]:


# Extract archives and prepare splits

split_frames = prepare_all_split_dataframes(DATA_ROOT, require_existing_files=True)

for split_name, frame in split_frames.items():
    print(split_name, frame.shape, frame['LABEL'].value_counts().to_dict())


def get_features(file_path):
    safe_name = file_path.replace("/", "_").replace(":", "_")
    cache_path = str(CACHE_DIR / (safe_name + ".pkl"))

    # Always recompute and overwrite
    try:
        feats = extract_prosodic_features(file_path)
        joblib.dump(feats, cache_path)
        return feats
    except Exception as e:
        print(f"Failed: {file_path} — {e}")
        return {col: 0.0 for col in PROSODIC_FEATURE_COLUMNS}


def extract_split(df, split_name):
    print(f"\nExtracting {split_name} — {len(df)} files using {N_JOBS} workers...")
    file_paths = df["FULL_FILE_PATH"].tolist()

    results = Parallel(n_jobs=N_JOBS, verbose=5)(
        delayed(get_features)(fp) for fp in file_paths
    )

    print(f"Done {split_name}!")
    return results


# Extract all splits
for split_name, df in split_frames.items():
    extract_split(df, split_name)

print("\nAll features extracted and cached successfully!")
print(f"Cache location: {CACHE_DIR}")